TALLER NLP

Integrantes:
- Jaiber Antonio Leon Garcia
- Manuel Garzon Tovar
URl: https://colab.research.google.com/drive/1oT2fubSBRnixTOQoLJwMFlNwd7CFVTXV?usp=sharing

In [ ]:
!pip install nltk scikit-learn numpy

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
import re
import unicodedata
from collections import Counter
import numpy as np
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

STOPWORDS_ES = set(stopwords.words('spanish'))

Ejercicio 1

Crea una función limpiar_texto() que convierta a minúsculas, remueva URLs, emails, caracteres especiales y normalice espacios en blanco.

In [ ]:
def limpiar_texto(texto):
    texto = texto.lower()
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    texto = re.sub(r'\S+@\S+\.\S+', '', texto)
    texto = re.sub(r'[^a-záéíóúüñ\s]', '', texto)
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

textos_prueba = [
    "¡Hola!!! Visita https://ejemplo.com o envía tu CV a recurso@empresa.com. Hay   espacios extra!!!",
    "Python3.9 es excelente!!! Descarga desde www.python.org #programming @user",
    "El NLP es fascinante... ¿No lo crees? (sí/no) [responder]"
]
for i, texto in enumerate(textos_prueba, 1):
    print(f"Texto {i}:")
    print(f"  Original : {texto}")
    print(f"  Limpio   : {limpiar_texto(texto)}\n")

Texto 1:
  Original : ¡Hola!!! Visita https://ejemplo.com o envía tu CV a recurso@empresa.com. Hay   espacios extra!!!
  Limpio   : hola visita o envía tu cv a hay espacios extra

Texto 2:
  Original : Python3.9 es excelente!!! Descarga desde www.python.org #programming @user
  Limpio   : python es excelente descarga desde programming user

Texto 3:
  Original : El NLP es fascinante... ¿No lo crees? (sí/no) [responder]
  Limpio   : el nlp es fascinante no lo crees síno responder



Ejercicio 2

Implementa la función analizar_tokenizacion() que divida un texto en tokens de palabras y oraciones usando NLTK, y retorne el conteo de cada uno.

In [ ]:
from nltk.tokenize import word_tokenize, sent_tokenize

def analizar_tokenizacion(texto, idioma='spanish'):
    return {
        'tokens_palabra': [t for t in word_tokenize(texto.lower(), language=idioma) if t.isalpha()],
        'tokens_oracion': sent_tokenize(texto, language=idioma),
        'num_palabras': len([t for t in word_tokenize(texto.lower(), language=idioma) if t.isalpha()]),
        'num_oraciones': len(sent_tokenize(texto, language=idioma))
    }

for i, texto in enumerate([
    "Hola. ¿Cómo estás? Estoy muy bien, gracias por preguntar.",
    "El procesamiento del lenguaje natural es increíble! ¿No crees? Sí, totalmente."
], 1):
    r = analizar_tokenizacion(texto)
    print(f"Texto {i}: {texto}")
    print(f"  Palabras  : {r['tokens_palabra']}")
    print(f"  Oraciones : {r['tokens_oracion']}")
    print(f"  Total     : {r['num_palabras']} palabras | {r['num_oraciones']} oraciones\n")

Texto 1: Hola. ¿Cómo estás? Estoy muy bien, gracias por preguntar.
  Palabras  : ['hola', 'estás', 'estoy', 'muy', 'bien', 'gracias', 'por', 'preguntar']
  Oraciones : ['Hola.', '¿Cómo estás?', 'Estoy muy bien, gracias por preguntar.']
  Total     : 8 palabras | 3 oraciones

Texto 2: El procesamiento del lenguaje natural es increíble! ¿No crees? Sí, totalmente.
  Palabras  : ['el', 'procesamiento', 'del', 'lenguaje', 'natural', 'es', 'increíble', 'crees', 'sí', 'totalmente']
  Oraciones : ['El procesamiento del lenguaje natural es increíble!', '¿No crees?', 'Sí, totalmente.']
  Total     : 10 palabras | 3 oraciones



Ejercicio 3

Crea la función normalizar_texto() que remueva acentos y caracteres diacríticos usando unicodedata, convirtiendo por ejemplo "Café" en "cafe".

In [ ]:
def normalizar_texto(texto):
    texto = texto.lower()
    texto = unicodedata.normalize('NFD', texto)
    return ''.join(c for c in texto if unicodedata.category(c) != 'Mn')

for texto in ["¡Hola! ¿Cómo Estás?", "Programación y árboles de decisión", "La física y química de la Tierra", "Café con leche es delicioso"]:
    print(f"  Original    : {texto}")
    print(f"  Normalizado : {normalizar_texto(texto)}\n")

  Original    : ¡Hola! ¿Cómo Estás?
  Normalizado : ¡hola! ¿como estas?

  Original    : Programación y árboles de decisión
  Normalizado : programacion y arboles de decision

  Original    : La física y química de la Tierra
  Normalizado : la fisica y quimica de la tierra

  Original    : Café con leche es delicioso
  Normalizado : cafe con leche es delicioso



Ejercicio 4

Implementa remover_stopwords() que filtre las palabras comunes del español (el, la, de, que...) de un texto y retorne solo las palabras con significado.

In [ ]:
def remover_stopwords(texto):
    tokens = re.findall(r'\b\w+\b', texto.lower())
    return [t for t in tokens if t not in STOPWORDS_ES and len(t) > 1]

for texto in [
    "El procesamiento del lenguaje natural es fascinante y útil",
    "La inteligencia artificial está transformando el mundo",
    "Los algoritmos de aprendizaje automático son muy poderosos"
]:
    resultado = remover_stopwords(texto)
    print(f"  Original      : {texto}")
    print(f"  Sin stopwords : {' '.join(resultado)}")
    print(f"  Removidas     : {len(texto.split()) - len(resultado)} palabras\n")

  Original      : El procesamiento del lenguaje natural es fascinante y útil
  Sin stopwords : procesamiento lenguaje natural fascinante útil
  Removidas     : 4 palabras

  Original      : La inteligencia artificial está transformando el mundo
  Sin stopwords : inteligencia artificial transformando mundo
  Removidas     : 3 palabras

  Original      : Los algoritmos de aprendizaje automático son muy poderosos
  Sin stopwords : algoritmos aprendizaje automático poderosos
  Removidas     : 4 palabras



Ejercicio 5

Compara stemming y lemmatización: usa SnowballStemmer para reducir palabras en español a su raíz y analiza la diferencia entre ambas técnicas.

In [ ]:
stemmer = SnowballStemmer('spanish')
palabras_es = ['correr','corriendo','corrió','corrida','programar','programación','programador','análisis','analizar','analizando']

print(f"{'Palabra':<20} {'Stem'}")
print("-" * 35)
for palabra in palabras_es:
    print(f"  {palabra:<18} {stemmer.stem(palabra)}")

Palabra              Stem
-----------------------------------
  correr             corr
  corriendo          corr
  corrió             corr
  corrida            corr
  programar          program
  programación       program
  programador        program
  análisis           analisis
  analizar           analiz
  analizando         analiz


Ejercicio 6

Crea obtener_palabras_clave() que cuente la frecuencia de palabras en un texto (sin stopwords) y retorne las N más comunes.

In [ ]:
def obtener_palabras_clave(texto, n=10):
    tokens = re.findall(r'\b[a-záéíóúüñ]+\b', texto.lower())
    tokens_filtrados = [t for t in tokens if t not in STOPWORDS_ES and len(t) >= 2]
    return Counter(tokens_filtrados).most_common(n)

texto_analisis = """
La Inteligencia Artificial es revolucionaria. La IA transforma industrias.
El aprendizaje automático es clave. Los modelos de IA mejoran constantemente.
Los algoritmos de machine learning procesan datos masivos.
La IA encontrada en aplicaciones modernas es muy sofisticada.
Programación y IA son futuro del software.
"""

print(f"{'#':<4} {'Palabra':<20} {'Freq':<6} {'Barra'}")
print("-" * 50)
for i, (palabra, freq) in enumerate(obtener_palabras_clave(texto_analisis, 8), 1):
    print(f"  {i:<2} {palabra:<20} {freq:<6} {'█' * freq * 5}")

#    Palabra              Freq   Barra
--------------------------------------------------
  1  ia                   4      ████████████████████
  2  inteligencia         1      █████
  3  artificial           1      █████
  4  revolucionaria       1      █████
  5  transforma           1      █████
  6  industrias           1      █████
  7  aprendizaje          1      █████
  8  automático           1      █████


Ejercicio 7

Implementa extraer_entidades() que identifique y clasifique entidades nombradas en un texto: personas, organizaciones, lugares, dinero y fechas.

In [ ]:
PERSONAS_REF = ['juan','maría','pedro','ana','carlos','garcía','lópez']
ORGS_REF     = ['google','microsoft','apple','amazon','meta','openai']
LUGARES_REF  = ['bogotá','madrid','colombia','españa','medellín','cali']

def extraer_entidades(texto):
    entidades, texto_lower = [], texto.lower()
    for lista, tipo in [(PERSONAS_REF,'PERSONA'),(ORGS_REF,'ORGANIZACIÓN'),(LUGARES_REF,'LUGAR')]:
        for nombre in lista:
            for m in re.finditer(re.escape(nombre), texto_lower):
                entidades.append({'texto': texto[m.start():m.end()].title(), 'tipo': tipo, 'pos': m.start()})
    for m in re.finditer(r'\d+\s*(millones?|dólares?|pesos?)', texto, re.IGNORECASE):
        entidades.append({'texto': m.group().strip(), 'tipo': 'DINERO', 'pos': m.start()})
    for m in re.finditer(r'\b(19|20)\d{2}\b', texto):
        entidades.append({'texto': m.group(), 'tipo': 'FECHA', 'pos': m.start()})
    vistas, resultado = set(), []
    for e in sorted(entidades, key=lambda x: x['pos']):
        if (e['texto'].lower(), e['tipo']) not in vistas:
            vistas.add((e['texto'].lower(), e['tipo']))
            resultado.append(e)
    return resultado

print(f"  {iconos.get(e['tipo'],'•')} [{e['tipo']}] → '{e['texto']}'")
for i, texto in enumerate([
    "Juan García, director de Google, anunció en Bogotá una inversión de 50 millones de dólares.",
    "María López trabaja en Microsoft desde el 2020 y reside en Medellín, Colombia."
], 1):
    print(f"Texto {i}: {texto}")
    for e in extraer_entidades(texto):
       print(f"  [{e['tipo']}] → '{e['texto']}'")
    print()

  📍 [LUGAR] → 'Colombia'
Texto 1: Juan García, director de Google, anunció en Bogotá una inversión de 50 millones de dólares.
  [PERSONA] → 'Juan'
  [PERSONA] → 'García'
  [ORGANIZACIÓN] → 'Google'
  [LUGAR] → 'Bogotá'
  [DINERO] → '50 millones'

Texto 2: María López trabaja en Microsoft desde el 2020 y reside en Medellín, Colombia.
  [PERSONA] → 'María'
  [PERSONA] → 'López'
  [ORGANIZACIÓN] → 'Microsoft'
  [FECHA] → '2020'
  [LUGAR] → 'Medellín'
  [LUGAR] → 'Colombia'



Ejercicio 8

Crea calcular_similitud_textos() que use TF-IDF y similitud coseno para comparar qué tan similares son varios textos entre sí, retornando una matriz de similitud.

In [ ]:
def calcular_similitud_textos(textos):
    return cosine_similarity(TfidfVectorizer().fit_transform(textos))

textos_similitud = [
    "El procesamiento del lenguaje natural es importante para la IA",
    "NLP y procesamiento de texto son relevantes en inteligencia artificial",
    "Los gatos son animales domésticos muy populares",
    "La comprensión del lenguaje natural mediante máquinas es fundamental",
    "Los perros y gatos son mascotas comunes en los hogares"
]

matriz = calcular_similitud_textos(textos_similitud)
etiquetas = [f"T{i+1}" for i in range(len(textos_similitud))]
print(f"{'':>5}", end='')
for e in etiquetas: print(f"{e:>8}", end='')
print()
for i, fila in enumerate(matriz):
    print(f"{etiquetas[i]:>5}", end='')
    for val in fila: print(f"{val:>8.3f}", end='')
    print()
print()
for i, t in enumerate(textos_similitud): print(f"  T{i+1}: {t}")

           T1      T2      T3      T4      T5
   T1   1.000   0.083   0.000   0.430   0.000
   T2   0.083   1.000   0.067   0.000   0.137
   T3   0.000   0.067   1.000   0.000   0.346
   T4   0.430   0.000   0.000   1.000   0.000
   T5   0.000   0.137   0.346   0.000   1.000

  T1: El procesamiento del lenguaje natural es importante para la IA
  T2: NLP y procesamiento de texto son relevantes en inteligencia artificial
  T3: Los gatos son animales domésticos muy populares
  T4: La comprensión del lenguaje natural mediante máquinas es fundamental
  T5: Los perros y gatos son mascotas comunes en los hogares


Ejercicio 9

Implementa la clase PreprocessadorTexto con un método procesar() que integre en un pipeline completo: limpieza, normalización, tokenización, remoción de stopwords y stemming.

In [ ]:
class PreprocessadorTexto:
    def __init__(self):
        self.stemmer = SnowballStemmer('spanish')

    def _limpiar(self, texto):
        texto = texto.lower()
        texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
        texto = re.sub(r'\S+@\S+\.\S+', '', texto)
        texto = re.sub(r'[^a-záéíóúüñ\s]', '', texto)
        return re.sub(r'\s+', ' ', texto).strip()

    def _normalizar(self, texto):
        return ''.join(c for c in unicodedata.normalize('NFD', texto) if unicodedata.category(c) != 'Mn')

    def _tokenizar(self, texto):
        return re.findall(r'\b\w+\b', texto)

    def _remover_stopwords(self, tokens):
        return [t for t in tokens if t not in STOPWORDS_ES and len(t) > 1]

    def procesar(self, texto):
        pasos = {'original': texto}
        pasos['limpio']        = self._limpiar(texto)
        pasos['normalizado']   = self._normalizar(pasos['limpio'])
        pasos['tokens']        = self._tokenizar(pasos['normalizado'])
        pasos['sin_stopwords'] = self._remover_stopwords(pasos['tokens'])
        pasos['stems']         = [self.stemmer.stem(t) for t in pasos['sin_stopwords']]
        return pasos

pipeline = PreprocessadorTexto()
for i, texto in enumerate([
    "¡Visita https://nlp.com! El NLP es fascinante. Envía: info@test.com",
    "Los algoritmos de Aprendizaje Automático transforman la Inteligencia Artificial!",
    "Programación, análisis y procesamiento de datos son habilidades clave en 2024."
], 1):
    print(f"--- Texto {i} ---")
    for etapa, valor in pipeline.procesar(texto).items():
        print(f"  {etapa:<18}: {valor}")
    print()

--- Texto 1 ---
  original          : ¡Visita https://nlp.com! El NLP es fascinante. Envía: info@test.com
  limpio            : visita el nlp es fascinante envía
  normalizado       : visita el nlp es fascinante envia
  tokens            : ['visita', 'el', 'nlp', 'es', 'fascinante', 'envia']
  sin_stopwords     : ['visita', 'nlp', 'fascinante', 'envia']
  stems             : ['visit', 'nlp', 'fascin', 'envi']

--- Texto 2 ---
  original          : Los algoritmos de Aprendizaje Automático transforman la Inteligencia Artificial!
  limpio            : los algoritmos de aprendizaje automático transforman la inteligencia artificial
  normalizado       : los algoritmos de aprendizaje automatico transforman la inteligencia artificial
  tokens            : ['los', 'algoritmos', 'de', 'aprendizaje', 'automatico', 'transforman', 'la', 'inteligencia', 'artificial']
  sin_stopwords     : ['algoritmos', 'aprendizaje', 'automatico', 'transforman', 'inteligencia', 'artificial']
  stems             : 

Ejercicio 10

Crea la clase AnalizadorSentimientos que clasifique textos como POSITIVO, NEGATIVO o NEUTRO usando similitud TF-IDF contra listas de palabras clave de cada sentimiento.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
import unicodedata

class AnalizadorSentimientos:
    """
    Analizador simple de sentimientos basado en similitud de textos.
    """

    def __init__(self):
        self.palabras_positivas = [
            'excelente increible fantastico maravilloso perfecto genial',
            'feliz contento satisfecho bueno encanta recomiendo util',
            'rapido eficiente profesional facil calidad brillante positivo'
        ]
        self.palabras_negativas = [
            'terrible horrible malo pesimo deficiente decepcionante',
            'frustrado molesto odio lento ineficiente nunca mas',
            'problema error falla defecto arruinado roto negativo'
        ]
        self.palabras_neutras = [
            'normal regular comun estandar basico promedio',
            'moderado aceptable razonable cumple funciona'
        ]

    def _preprocesar(self, texto):
        texto = texto.lower()
        texto = unicodedata.normalize('NFD', texto)
        texto = ''.join(c for c in texto if unicodedata.category(c) != 'Mn')
        texto = re.sub(r'[^a-z\s]', '', texto)
        return ' '.join(t for t in texto.split() if t not in STOPWORDS_ES and len(t) > 1)

    def _extraer_palabras_clave(self, texto, tipo):
        proc = self._preprocesar(texto)
        if tipo == 'positivo':
            referencia = ' '.join(self.palabras_positivas)
        elif tipo == 'negativo':
            referencia = ' '.join(self.palabras_negativas)
        else:
            referencia = ' '.join(self.palabras_neutras)
        tokens_texto = set(proc.split())
        tokens_ref   = set(referencia.split())
        return list(tokens_texto & tokens_ref)

    def analizar(self, texto):
        proc = self._preprocesar(texto)
        todos = self.palabras_positivas + self.palabras_negativas + self.palabras_neutras + [proc]
        np_ = len(self.palabras_positivas)
        nn  = len(self.palabras_negativas)
        nne = len(self.palabras_neutras)

        try:
            tfidf = TfidfVectorizer().fit_transform(todos)
            v   = tfidf[-1]
            sp  = float(cosine_similarity(v, tfidf[:np_]).mean())
            sn  = float(cosine_similarity(v, tfidf[np_:np_+nn]).mean())
            sne = float(cosine_similarity(v, tfidf[np_+nn:-1]).mean())
        except:
            sp = sn = sne = 0.0

        max_sim = max(sp, sn, sne)
        if max_sim < 0.005:
            sentimiento = 'NEUTRAL'
        elif sp >= sn and sp >= sne:
            sentimiento = 'POSITIVO'
        elif sn >= sp and sn >= sne:
            sentimiento = 'NEGATIVO'
        else:
            sentimiento = 'NEUTRAL'

        palabras_clave = self._extraer_palabras_clave(texto, sentimiento.lower())

        resultado = {
            'texto':               texto,
            'sentimiento':         sentimiento,
            'confianza':           round(max_sim, 4),
            'similitud_positiva':  round(sp, 4),
            'similitud_negativa':  round(sn, 4),
            'similitud_neutral':   round(sne, 4),
            'palabras_clave':      palabras_clave
        }
        return resultado


# Textos de prueba
opiniones = [
    "Este producto es excelente y me encanta. Muy recomendado!",
    "Terriblemente malo. No funcionó. Decepcionante y frustrante.",
    "El producto llegó a tiempo. Normal, sin sorpresas.",
    "¡Increíble! Superó todas mis expectativas. Fantástico!",
    "Horrible, defectuoso y de mala calidad. Totalmente decepción."
]

print("Análisis de Sentimientos")
print("=" * 80)
print(f"{'Opinion':<52} {'Sentimiento':<12} {'Confianza':>10}")
print("-" * 80)

analizador = AnalizadorSentimientos()
resultados = []

for opinion in opiniones:
    r = analizador.analizar(opinion)
    resultados.append(r)
    corta = opinion[:49] + '...' if len(opinion) > 49 else opinion
    print(f"  {corta:<50} {r['sentimiento']:<12} {r['confianza']:>10.4f}")

print("\nDetalle por opinion:")
print("=" * 80)
for r in resultados:
    print(f"\nTexto     : {r['texto']}")
    print(f"Sentimiento       : {r['sentimiento']}")
    print(f"Confianza         : {r['confianza']}")
    print(f"Sim. positiva     : {r['similitud_positiva']}")
    print(f"Sim. negativa     : {r['similitud_negativa']}")
    print(f"Sim. neutral      : {r['similitud_neutral']}")
    print(f"Palabras clave    : {r['palabras_clave'] if r['palabras_clave'] else 'ninguna detectada'}")

Análisis de Sentimientos
Opinion                                              Sentimiento   Confianza
--------------------------------------------------------------------------------
  Este producto es excelente y me encanta. Muy reco... POSITIVO         0.1033
  Terriblemente malo. No funcionó. Decepcionante y ... NEGATIVO         0.0970
  El producto llegó a tiempo. Normal, sin sorpresas... NEUTRAL          0.0687
  ¡Increíble! Superó todas mis expectativas. Fantás... POSITIVO         0.0970
  Horrible, defectuoso y de mala calidad. Totalment... NEGATIVO         0.0427

Detalle por opinion:

Texto     : Este producto es excelente y me encanta. Muy recomendado!
Sentimiento       : POSITIVO
Confianza         : 0.1033
Sim. positiva     : 0.1033
Sim. negativa     : 0.0
Sim. neutral      : 0.0
Palabras clave    : ['encanta', 'excelente']

Texto     : Terriblemente malo. No funcionó. Decepcionante y frustrante.
Sentimiento       : NEGATIVO
Confianza         : 0.097
Sim. positiva     : 0.0
